# Hasil eksekusi Colab: pertemuan3_pengkodisian_perulangan

Dijalankan di VM Colab GPU T4 (sesi `d920cd`). Sel di bawah adalah kode notebook asli beserta outputnya.


In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/ray-project/raydp/master/tutorials/dataset/healthcare-dataset-stroke-data.csv"
df = pd.read_csv(URL)

print("Jumlah baris :", df.shape[0])
print("Jumlah kolom :", df.shape[1])
df.head(3)

Jumlah baris : 5110
Jumlah kolom : 12


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1


In [ ]:
# Contoh slide, tetap jalan
akurasi = 0.87
print(akurasi >= 0.80)  # True

label = "batik"
print(label in ["batik", "endek"])  # True

nilai = None
print(nilai is None)  # True

# Terapkan ke dataset
print("bmi kosong       :", int(df["bmi"].isna().sum()))
print("smoking Unknown  :", int((df["smoking_status"] == "Unknown").sum()))
print("bmi di atas 60   :", int((df["bmi"] > 60).sum()))

# Gabungan and / or / not pada satu baris
mask = (df["age"] >= 60) & (df["hypertension"] == 1) & (df["smoking_status"] != "Unknown")
print("lansia + hipertensi + smoking diketahui:", int(mask.sum()))

assert int(mask.sum()) > 0  # ponytail: cek kewarasan, bukan uji statistik

True
True
True
bmi kosong       : 201
smoking Unknown  : 1544
bmi di atas 60   : 13
lansia + hipertensi + smoking diketahui: 261


In [ ]:
def kategori_glukosa(g):
    if g >= 200:      # paling ketat dulu
        return "sangat tinggi"
    elif g >= 140:
        return "tinggi"
    elif g >= 100:
        return "waspada"
    else:
        return "normal"


def kategori_risiko(age, hyp, heart):
    if heart == 1 and hyp == 1:
        return "tinggi"
    elif heart == 1 or hyp == 1:
        return "sedang"
    elif age >= 60:
        return "waspada"
    else:
        return "rendah"


# Uji batas: bawah, pas, atas
for g in [99.9, 100.0, 139.9, 140.0, 199.9, 200.0]:
    print(g, "->", kategori_glukosa(g))

print()
for i in range(5):
    r = df.iloc[i]
    print(
        f"baris {i}: glukosa={r['avg_glucose_level']:.1f} "
        f"({kategori_glukosa(r['avg_glucose_level'])}) "
        f"risiko={kategori_risiko(r['age'], r['hypertension'], r['heart_disease'])}"
    )

99.9 -> normal
100.0 -> waspada
139.9 -> waspada
140.0 -> tinggi
199.9 -> tinggi
200.0 -> sangat tinggi

baris 0: glukosa=228.7 (sangat tinggi) risiko=sedang
baris 1: glukosa=202.2 (sangat tinggi) risiko=waspada
baris 2: glukosa=105.9 (waspada) risiko=sedang
baris 3: glukosa=171.2 (tinggi) risiko=rendah
baris 4: glukosa=174.1 (tinggi) risiko=sedang


In [ ]:
# enumerate: indeks + nilai
for indeks, skor in enumerate([78, 85, 91]):
    print(indeks, skor)

print()
# range: epoch ala slide
for epoch in range(1, 6):
    print("epoch", epoch)

print()
# enumerate + zip pada 5 pasien pertama
usia = df["age"].head(5).tolist()
bmi = df["bmi"].head(5).tolist()
for i, (u, b) in enumerate(zip(usia, bmi)):
    print(f"pasien {i}: usia={u} bmi={b}")

# for untuk agregasi sederhana, dicek melawan pandas
total, n = 0.0, 0
for v in df["age"].head(100):
    total += v
    n += 1
rata_loop = total / n
rata_pandas = df["age"].head(100).mean()
print("\nrata-rata usia 100 baris (loop):", round(rata_loop, 4))
assert abs(rata_loop - rata_pandas) < 1e-9

0 78
1 85
2 91

epoch 1
epoch 2
epoch 3
epoch 4
epoch 5

pasien 0: usia=67.0 bmi=36.6
pasien 1: usia=61.0 bmi=nan
pasien 2: usia=80.0 bmi=32.5
pasien 3: usia=49.0 bmi=34.4
pasien 4: usia=79.0 bmi=24.0

rata-rata usia 100 baris (loop): 67.28


In [ ]:
# Contoh slide: loss menyusut tiap epoch
epoch, loss = 1, 1.0
while epoch <= 10 and loss > 0.10:
    loss *= 0.72
    print(epoch, round(loss, 3))
    epoch += 1  # tanpa baris ini loop tidak pernah berhenti

print()
# while pada dataset: kumpulkan 5 pasien berbmi valid pertama
ketemu, i = [], 0
while len(ketemu) < 5 and i < len(df):
    b = df.iloc[i]["bmi"]
    if pd.notna(b):
        ketemu.append((i, b))
    i += 1
print("5 bmi valid pertama (indeks, bmi):", ketemu)
assert len(ketemu) == 5

1 0.72
2 0.518
3 0.373
4 0.269
5 0.193
6 0.139
7 0.1
8 0.072

5 bmi valid pertama (indeks, bmi): [(0, np.float64(36.6)), (2, np.float64(32.5)), (3, np.float64(34.4)), (4, np.float64(24.0)), (5, np.float64(29.0))]


In [ ]:
# Contoh slide
for nilai in [12, None, 18, -3, 25]:
    if nilai is None:
        continue
    if nilai < 0:
        break
    print(nilai)

print()
# Versi dataset: pindai 20 bmi pertama
# Aturan demo: NaN dilewati (continue), bmi > 90 hentikan demo (break)
diproses, dilewati = 0, 0
for b in df["bmi"].head(20).tolist():
    if pd.isna(b):
        dilewati += 1
        continue
    if b > 90:
        print(f"bmi ekstrem {b}, hentikan pindaian demo")
        break
    diproses += 1
print(f"diproses={diproses} dilewati(NaN)={dilewati}")

12
18

diproses=16 dilewati(NaN)=4


In [ ]:
def kelompok_usia(a):
    if a < 18:
        return "anak (<18)"
    elif a < 60:
        return "dewasa (18-59)"
    else:
        return "lansia (>=60)"


hasil = []
for ku in ["anak (<18)", "dewasa (18-59)", "lansia (>=60)"]:
    for hyp in [0, 1]:
        pot = df[(df["age"].apply(kelompok_usia) == ku) & (df["hypertension"] == hyp)]
        rate = pot["stroke"].mean() if len(pot) else 0.0
        hasil.append((ku, hyp, len(pot), round(float(rate), 4)))
        print(f"{ku} x hipertensi={hyp}: n={len(pot)} stroke_rate={rate:.4f}")

total = sum(h[2] for h in hasil)
print("\ntotal baris terpetakan:", total)
assert total == len(df)  # tiap baris masuk tepat satu sel

anak (<18) x hipertensi=0: n=855 stroke_rate=0.0023
anak (<18) x hipertensi=1: n=1 stroke_rate=0.0000
dewasa (18-59) x hipertensi=0: n=2677 stroke_rate=0.0198
dewasa (18-59) x hipertensi=1: n=201 stroke_rate=0.0647
lansia (>=60) x hipertensi=0: n=1080 stroke_rate=0.1185
lansia (>=60) x hipertensi=1: n=296 stroke_rate=0.1791

total baris terpetakan: 5110


In [ ]:
# Contoh slide
data = [10, None, 20, 30]
bersih = [x for x in data if x is not None]
skala = [x / 100 for x in bersih]
print(bersih, skala)

# Versi dataset: 10 bmi pertama
bmi_10 = df["bmi"].head(10).tolist()
bersih_bmi = [x for x in bmi_10 if pd.notna(x)]
skala_bmi = [x / 100 for x in bersih_bmi]
print("mentah :", bmi_10)
print("bersih :", bersih_bmi)
print("skala  :", [round(v, 4) for v in skala_bmi])
assert bersih_bmi == df["bmi"].head(10).dropna().tolist()

[10, 20, 30] [0.1, 0.2, 0.3]
mentah : [36.6, nan, 32.5, 34.4, 24.0, 29.0, 27.4, 22.8, nan, 24.2]
bersih : [36.6, 32.5, 34.4, 24.0, 29.0, 27.4, 22.8, 24.2]
skala  : [0.366, 0.325, 0.344, 0.24, 0.29, 0.274, 0.228, 0.242]


In [ ]:
data_valid = []
alasan = {"bmi_kosong": 0, "smoking_unknown": 0, "gender_other": 0, "bmi_negatif": 0}

for _, baris in df.iterrows():
    if pd.isna(baris["bmi"]):
        alasan["bmi_kosong"] += 1
        continue
    if baris["smoking_status"] == "Unknown":
        alasan["smoking_unknown"] += 1
        continue
    if baris["gender"] == "Other":
        alasan["gender_other"] += 1
        continue
    if baris["bmi"] < 0:  # tidak ada di dataset ini, cabang disiapkan ikut slide
        alasan["bmi_negatif"] += 1
        continue
    data_valid.append(baris)

n_valid = len(data_valid)
n_tolak = sum(alasan.values())
print(f"valid: {n_valid} ({n_valid/len(df)*100:.1f}%)")
print(f"ditolak: {n_tolak} ({n_tolak/len(df)*100:.1f}%)")
print(alasan)

# ponytail: tiap baris tepat satu nasib (valid atau satu alasan), maksimal cek tumpang tindih
assert n_valid + n_tolak == len(df)
assert alasan["bmi_negatif"] == 0  # dokumentasi: dataset ini memang tanpa bmi negatif

valid: 3425 (67.0%)
ditolak: 1685 (33.0%)
{'bmi_kosong': 201, 'smoking_unknown': 1483, 'gender_other': 1, 'bmi_negatif': 0}
